<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/etapa-01-logica/05_validacao_tautologia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmo de Validação da Tabela de Verdade e Prova Computacional de Tautologia

Este notebook executa a **validação algorítmica por exaustão (Prova por Tabela de Verdade)** da lógica de intertravamento do **SCADA-Core do AGV**. O objetivo é demonstrar que as regras de controle impedem matematicamente qualquer condição de risco operacional.

---

## 💡 Como o Código Funciona (Análise Etapa por Etapa)

### 1. Mapeamento do Espaço de Estados ($2^n$)
O sistema possui 5 entradas discretas ($c_1, d_1, g_1, b_1, s_1$). Usando a função `itertools.product([False, True], repeat=5)`, o script gera a matriz combinatória completa de $2^5 = 32$ linhas. Isso garante que **100% dos cenários operacionais possíveis** (normais, de falha isolada ou de falhas múltiplas simultâneas) sejam testados.

### 2. Simulação do Processamento em Cascata do CLP/SCADA
Para cada uma das 32 combinações, o código executa a lógica proposicional em ordem hierárquica:
* **Detecção de Falha Crítica ($F$):** Avalia se há parada de emergência ($s_1$), bateria crítica ($b_1$) ou vazamento de gás ($g_1$) via porta `OR`:
  $$F = s_1 \lor b_1 \lor g_1$$
* **Cálculo do Permissivo e Tração ($m_1$):** O motor só é energizado se houver operador ($c_1$), se ele não estiver perto demais ($\neg d_1$) **E** se não houver falha ($\neg F$) via porta `AND`:
  $$m_1 = P_{mov} = c_1 \land \neg d_1 \land \neg F$$
* **Sinalização de Alarme ($l_1$):** O alarme é ativado sempre que $F = 1$.

### 3. Teste do Estado de Risco Proibido ($S_{risco}$)
O código avalia continuamente a ocorrência do estado de risco não tolerado, definido pela intersecção entre uma falha ativa e a tração ligada:
$$S_{risco} \equiv F \land m_1$$

### 4. Prova Algorítmica da Tautologia (`.all()`)
A proposição de segurança afirma que o estado de risco é impossível, ou seja, $\neg S_{risco}$ deve ser sempre **Verdadeiro (1)**.

O método `.all()` da biblioteca Pandas varre a coluna `Tautologia (¬S_risco)` das 32 linhas. Se todas as 32 saídas forem iguais a `1` (True), o algoritmo confirma computacionalmente que o intertravamento é **estritamente seguro e à prova de falhas**.

In [4]:
import pandas as pd
from itertools import product

# 1. Definição do Espaço de Estados (Todas as combinações binárias das 5 entradas)
# Variaveis de entrada: c1 (Operador), d1 (Muito Perto), g1 (Gas), b1 (Bateria), s1 (E-Stop)
entradas = list(product([False, True], repeat=5))

tabela_dados = []

# 2. Processamento da Lógica do SCADA-Core para cada estado
for c1, d1, g1, b1, s1 in entradas:
    # A. Condição de Falha Crítica (F)
    F = s1 or b1 or g1

    # B. Permissivo de Movimento (P_mov)
    P_mov = c1 and (not d1) and (not F)

    # C. Estado do Motor de Tração (m1)
    m1 = P_mov

    # D. Estado do Alarme (l1)
    l1 = F

    # E. Teste do Estado de Risco Proibido (S_risco = Falha Ativa E Motor Ligado)
    S_risco = F and m1

    # F. Teorema de Segurança (Devance que S_risco é sempre FALSO -> Tautologia)
    Tautologia_Seguranca = not S_risco

    tabela_dados.append({
        'c1 (Operador)': int(c1),
        'd1 (Perto)': int(d1),
        'g1 (Gás)': int(g1),
        'b1 (Bateria)': int(b1),
        's1 (E-Stop)': int(s1),
        'F (Falha)': int(F),
        'P_mov (Permissivo)': int(P_mov),
        'm1 (Tração)': int(m1),
        'l1 (Alarme)': int(l1),
        'S_risco (F ∧ m1)': int(S_risco),
        'Tautologia (¬S_risco)': int(Tautologia_Seguranca)
    })

# 3. Criação do DataFrame pandas
df = pd.DataFrame(tabela_dados)

# 4. Validação Programática da Tautologia
eh_tautologia = df['Tautologia (¬S_risco)'].all()

# Exibição do Resultado no Colab
print("=" * 65)
print("     VALIDAÇÃO DE TAUTOLOGIA DE SEGURANÇA - SCADA CORE AGV     ")
print("=" * 65)

if eh_tautologia:
    print("\n✅ PROVA CONCLUÍDA COM SUCESSO:")
    print("   A proposição ¬S_risco é VERDADEIRA para 100% das 32 combinações lógicas.")
    print("   O motor de tração m1 NUNCA estará ativo durante uma falha crítica F.\n")
else:
    print("\n❌ FALHA DE SEGURANÇA ENCONTRADA:")
    print("   A matriz de intertravamento permite estados de risco!\n")

print("=" * 65)
print("TABELA DE VERDADE RESULTANTE (Visualização em formato Pandas):")
print("=" * 65)

# Exibe a tabela no Colab em formato visual
display(df)

     VALIDAÇÃO DE TAUTOLOGIA DE SEGURANÇA - SCADA CORE AGV     

✅ PROVA CONCLUÍDA COM SUCESSO:
   A proposição ¬S_risco é VERDADEIRA para 100% das 32 combinações lógicas.
   O motor de tração m1 NUNCA estará ativo durante uma falha crítica F.

TABELA DE VERDADE RESULTANTE (Visualização em formato Pandas):


,c1 (Operador),d1 (Perto),g1 (Gás),b1 (Bateria),s1 (E-Stop),F (Falha),P_mov (Permissivo),m1 (Tração),l1 (Alarme),S_risco (F ∧ m1),Tautologia (¬S_risco)
0,0,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,1,1,0,0,1,0,1
2,0,0,0,1,0,1,0,0,1,0,1
3,0,0,0,1,1,1,0,0,1,0,1
4,0,0,1,0,0,1,0,0,1,0,1
5,0,0,1,0,1,1,0,0,1,0,1
6,0,0,1,1,0,1,0,0,1,0,1
7,0,0,1,1,1,1,0,0,1,0,1
8,0,1,0,0,0,0,0,0,0,0,1
9,0,1,0,0,1,1,0,0,1,0,1
